# Memory Leak

A **CUDA memory leak in PyTorch** occurs when GPU memory is allocated for tensors, models, intermediate results, or computation graphs but is unintentionally retained instead of being released when no longer needed. Over time, GPU memory usage increases, potentially causing `CUDA out of memory (OOM)` errors.

In PyTorch, memory leaks commonly happen when you retain computation graphs, store tensors with gradients in Python lists, forget to detach tensors, or keep unnecessary references to GPU tensors. For example, `loss_history.append(loss)` can retain the computation graph for every training iteration. Instead, use `loss.item()` to store only the scalar value. Similarly, use `torch.no_grad()` or `torch.inference_mode()` during inference to avoid building unnecessary computation graphs. PyTorch automatically releases GPU allocations when tensors are no longer referenced, but its caching allocator may keep freed memory reserved for reuse.


## Memory leak during training

**1. Example: Memory leak during training**

**Incorrect**

```python
loss_history = []

for epoch in range(1000):
    output = model(input)
    loss = criterion(output, target)

    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    # Retains computation graph
    loss_history.append(loss)
```

**Problem:** Each stored `loss` can retain its computation graph and associated tensors, causing GPU memory consumption to grow.

**Correct**

```python
loss_history = []

for epoch in range(1000):
    output = model(input)
    loss = criterion(output, target)

    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    # Store only scalar value
    loss_history.append(loss.item())
```

**2. Common causes of CUDA memory leaks**

| Cause | Solution |
| :--- | :--- |
| Storing tensors with computation graphs | `tensor.detach()` |
| Storing loss tensors in lists | `loss.item()` |
| Unnecessary gradient calculation | `torch.no_grad()` |
| Keeping references to unused tensors | `del tensor` |
| Retaining computation graphs | Avoid unnecessary `retain_graph=True` |
| Accumulating gradients unintentionally | `optimizer.zero_grad()` |
| Keeping unnecessary GPU outputs | Move results to CPU using `.cpu()` |

---
### Different cases for Memory Leak

#### Case 1: Use of tensor.detach() how we can avoided memory leak
`tensor.detach()` helps prevent CUDA memory leaks by removing a tensor from the PyTorch computation graph while keeping its data accessible. When you store a tensor that has `requires_grad=True` in a Python list, PyTorch may retain its computation graph, including intermediate tensors required for backpropagation. This can keep GPU memory occupied unnecessarily.

Using `tensor.detach()` creates a new tensor that shares the same underlying memory but does not track gradients. Therefore, storing `loss.detach()` or `output.detach()` avoids retaining the original computation graph. For example, during training, use `loss_history.append(loss.detach().cpu())` if you need to retain tensor values for later analysis, or `loss_history.append(loss.item())` if you only need scalar loss values.

**1. Without `detach()` — Incorrect**

```python
loss_history = []

for epoch in range(100):
    output = model(input)
    loss = criterion(output, target)

    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    # Retains computation graph
    loss_history.append(loss)
```

**Problem:** `loss` is connected to the computation graph. Keeping references to it can retain graph-related memory across iterations.

**2. With `detach()` — Correct**

```python
loss_history = []

for epoch in range(100):
    output = model(input)
    loss = criterion(output, target)

    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    # Detach computation graph
    loss_history.append(loss.detach())
```
---
#### Case 2: tensor.cpu() Move tensor data to CPU & avoid problem memory leak

`tensor.detach()` helps prevent CUDA memory leaks by removing a tensor from the PyTorch computation graph while keeping its data accessible. When you store a tensor that has `requires_grad=True` in a Python list, PyTorch may retain its computation graph, including intermediate tensors required for backpropagation. This can keep GPU memory occupied unnecessarily.

Using `tensor.detach()` creates a new tensor that shares the same underlying memory but does not track gradients. Therefore, storing `loss.detach()` or `output.detach()` avoids retaining the original computation graph. For example, during training, use `loss_history.append(loss.detach().cpu())` if you need to retain tensor values for later analysis, or `loss_history.append(loss.item())` if you only need scalar loss values.

**1. Without `detach()` — Incorrect**

```python
loss_history = []

for epoch in range(100):
    output = model(input)
    loss = criterion(output, target)

    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    # Retains computation graph
    loss_history.append(loss)
```

**Problem:** `loss` is connected to the computation graph. Keeping references to it can retain graph-related memory across iterations.

**2. With `detach()` — Correct**

```python
loss_history = []

for epoch in range(100):
    output = model(input)
    loss = criterion(output, target)

    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    # Detach computation graph
    loss_history.append(loss.detach())
```
`tensor.cpu()` helps manage CUDA memory by moving tensor data from GPU memory (VRAM) to CPU memory (RAM). Once the tensor is moved, the new CPU tensor no longer occupies GPU memory. This is useful when you want to store intermediate outputs, predictions, loss values, or training results for later analysis without continuously consuming GPU memory.

For example, during training, if you store GPU outputs in a Python list, they remain on the GPU as long as references exist. Using `output.detach().cpu()` disconnects the output from the computation graph and transfers its data to CPU RAM. Once the original GPU tensor and other graph references are released, GPU memory can be reclaimed. Note that `.cpu()` alone does not detach the computation graph; use both APIs when appropriate.

**1. Without `.cpu()` — GPU memory retention**

```python
output_history = []

for epoch in range(100):
    output = model(input)

    # Stores GPU tensor
    output_history.append(output.detach())
```

**Problem:** `detach()` removes the computation graph, but the tensor still occupies GPU memory. Storing 100 outputs can consume significant VRAM.

**2. With `.cpu()` — Move data to RAM**

```python
output_history = []

for epoch in range(100):
    output = model(input)

    # Detach graph and move tensor to CPU
    output_history.append(output.detach().cpu())
```
Now:

* `detach()` disconnects the computation graph.
* `cpu()` transfers tensor data to CPU RAM.
* `output_history` stores CPU tensors instead of GPU tensors.
* GPU memory can be reclaimed when GPU references are no longer needed.
---



**3. How to debug CUDA memory leaks**

PyTorch provides several useful APIs:

```python
import torch

# Currently allocated GPU memory
print(torch.cuda.memory_allocated() / 1024**2, "MB")

# Total reserved GPU memory
print(torch.cuda.memory_reserved() / 1024**2, "MB")

# Peak allocated memory
print(torch.cuda.max_memory_allocated() / 1024**2, "MB")

# Memory statistics
print(torch.cuda.memory_summary())
```

Important distinction:

* `memory_allocated()` -> Memory occupied by live PyTorch tensors.
* `memory_reserved()` -> Memory managed by PyTorch's caching allocator.
* `torch.cuda.empty_cache()` -> Releases unused cached memory to the CUDA driver; it does not free memory occupied by live tensors.

**4. Interview question: How do you identify a CUDA memory leak?**

Monitor GPU memory across training iterations:

```python
for epoch in range(100):

    output = model(input)
    loss = criterion(output, target)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        allocated = torch.cuda.memory_allocated() / 1024**2
        reserved = torch.cuda.memory_reserved() / 1024**2

        print(
            f"Epoch: {epoch}, "
            f"Allocated: {allocated:.2f} MB, "
            f"Reserved: {reserved:.2f} MB"
        )
```

If `memory_allocated()` continuously increases across iterations even after temporary tensors should have been released, investigate retained tensor references and computation graphs.

**Important interview takeaway:** CUDA memory growth does not always indicate a memory leak. PyTorch's caching allocator intentionally retains unused memory. Distinguish between allocated and reserved memory before diagnosing a leak.